# Odor Vector — `recipe_odor_vectors.npy`

**Mission**: reproduce the recipe **odor vector** of the ICOCO paper from the new
`ICOCO.duckdb` pipeline (legacy reference:
`old_code/12_odor_vector/odor_vector_npy_export.ipynb`).

**Output**: `/workspace/7_create_npy_files/recipe_odor_vectors.npy`, dtype `float32`,
shape `(50511, 27)` — one row per recipe (`silver.RecipeZenodo` rowid order), one column per
odor (fixed 27-odor order, identical to the legacy notebook).

**Pipeline** (all legacy steps, SQL-ified)

1. **Cooking method → reactions**: the static `cooking_reactions` config is loaded from
   `/workspace/data/odor/reaction_cooking.py` (22 methods → 107 method-reaction pairs)
   into a TEMP table.
2. **(nutrient, reaction) → odors**: the LLM "gold" is derived **inside DuckDB** from the 3
   replica tables `odor.output1/2/3` with the same majority-vote (≥ 2 of 3, strip+lower)
   as `old_code/utils/gold_data_creator.py`. This was verified to reproduce the legacy
   `gold_reactions.csv` **exactly** (2,805 non-empty rows identical, 987 empty rows).
3. **Ingredient → nutrients**: legacy JSON
   `old_code/12_odor_vector/data/dirty_ingredient_nutrient_mapping.json`
   (no DuckDB copy exists; mission says *no new tables*, so it is read with `read_json`).
4. Per recipe: `nutrients × reactions` pairs → concatenated odor lists → frequency shares.
   The denominator counts **every** odor returned (even names outside the 27-odor axis —
   they get no column), faithful to the legacy code.

**Legacy deltas (expected, documented)**: legacy shape `(50513, 27)` (2 rows dropped in the
new bronze ingestion; alignment offsets at old rows **21475 / 23805**). `recipe_odor_entropy.npy`
(the per-recipe Shannon entropy of this matrix) is **also produced here — section 6**,
reproducing `old_code/12_odor_vector/odor_entropy_get.ipynb`.

## 0. Setup — connect → close (flush WAL) → reconnect

Same connection pattern as `silver_curation.ipynb` / `explore_zenodo_recipe.ipynb`:
close first so the WAL file (`ICOCO.duckdb.wal`) is flushed, then reconnect fresh.
This notebook only **reads** DuckDB and writes **TEMP** objects (session-local, vanish on
disconnect), so re-running it never changes the persistent database state.

In [1]:
%load_ext sql
%config SqlMagic.displaylimit = 100  # show all rows in query results
%sql duckdb:////workspace/ICOCO.duckdb

Connecting to 'duckdb:////workspace/ICOCO.duckdb'

In [2]:
# Close the connection FIRST to flush the WAL file (ICOCO.duckdb.wal)
# and avoid data corruption from an aborted transaction.
%sql --close duckdb:////workspace/ICOCO.duckdb

In [3]:
# Reconnect with a fresh connection (recovers/checkpoints the WAL)
%sql duckdb:////workspace/ICOCO.duckdb

Connecting and switching to connection 'duckdb:////workspace/ICOCO.duckdb'

In [4]:
from sql.connection import ConnectionManager

# IMPORTANT: DuckDB allows only ONE connection to the database at a time.
# Reuse the SAME connection that %sql already opened (do NOT duckdb.connect() again).
raw = ConnectionManager().current.connection_sqlalchemy.connection

# Keep RAM safe inside the 3.68 GB container: cap DuckDB memory + threads.
raw.execute("SET memory_limit = '1.5GB'")
raw.execute("SET threads = 4")
print('Reusing the single %sql connection for Python-side fetches (raw):', raw)

Reusing the single %sql connection for Python-side fetches (raw): <sqlalchemy.pool.base._ConnectionFairy object at 0x7ee89ff0dea0>


## 1. Static config → TEMP tables

`cooking_reactions` comes from the canonical config file (not hard-coded), and the fixed
27-odor axis matches the legacy notebook exactly.

In [5]:
# ---- 1. cooking method -> reactions, from the canonical config file ----
_ns = {}
exec(open('/workspace/data/odor/reaction_cooking.py').read(), _ns)
cooking_reactions = _ns['cooking_reactions']
method_rxn = sorted({(m.strip().lower(), r.strip().lower())
                     for m, rs in cooking_reactions.items() for r in rs})
print('cooking methods:', len(cooking_reactions), '| method-reaction pairs:', len(method_rxn))

# ---- 2. the fixed 27-odor axis (same order as the legacy notebook) ----
ODORS = ['vegetable', 'herbaceous', 'fatty', 'spicy', 'floral', 'balsamic', 'citrus',
         'woody', 'meaty', 'green', 'wine_like', 'earthy', 'fruity', 'chemical', 'nutty',
         'camphoraceous', 'tobacco', 'minty', 'pepper', 'medicinal', 'mossy', 'fishy',
         'smoky', 'musky', 'soapy', 'seedy', 'animal']
assert len(ODORS) == 27

# ---- 3. TEMP lookup tables (session-local, idempotent) ----
raw.execute('CREATE OR REPLACE TEMP TABLE cooking_reactions(method VARCHAR, reaction VARCHAR)')
raw.executemany('INSERT INTO cooking_reactions VALUES (?, ?)', method_rxn)
raw.execute('CREATE OR REPLACE TEMP TABLE odor_axis(idx INTEGER, odor VARCHAR)')
raw.executemany('INSERT INTO odor_axis VALUES (?, ?)', list(enumerate(ODORS)))
print('TEMP tables ready: cooking_reactions, odor_axis')

cooking methods: 22 | method-reaction pairs: 107


TEMP tables ready: cooking_reactions, odor_axis


## 2. Build the (nutrient, reaction) → odor gold map in SQL

Majority vote ≥ 2/3 over `odor.output1/2/3` — verified below against the legacy
`gold_reactions.csv`.

In [6]:
%%sql
-- (nutrient, reaction) -> odor gold map, majority vote >= 2 of 3 replicas.
-- Same normalization as old_code/utils/gold_data_creator.py: strip + lower each
-- element, count occurrences across the 3 lists, keep those seen >= 2 times.
-- The input text is '<Nutrient> AND <Reaction>'; the greedy regex splits at the
-- LAST ' AND ' (equivalent to the legacy rsplit(' AND ', 1)).
CREATE OR REPLACE TEMP VIEW odor_gold_map AS
WITH runs AS (
    SELECT custom_id, input_text, value_list FROM odor.output1
    UNION ALL SELECT custom_id, input_text, value_list FROM odor.output2
    UNION ALL SELECT custom_id, input_text, value_list FROM odor.output3
),
elems AS (
    SELECT custom_id,
           any_value(input_text) AS input_text,
           lower(trim(e))        AS odor,
           count(*)              AS n
    FROM runs, UNNEST(value_list) AS u(e)
    GROUP BY custom_id, lower(trim(e))
),
gold AS (
    SELECT custom_id, input_text, odor
    FROM elems
    WHERE n >= 2
)
SELECT custom_id,
       lower(trim(regexp_extract(input_text, '^(.*) AND (.*)$', 1))) AS nutrient,
       lower(trim(regexp_extract(input_text, '^(.*) AND (.*)$', 2))) AS reaction,
       odor
FROM gold
WHERE contains(input_text, ' AND ');

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [7]:
%%sql
SELECT count(DISTINCT custom_id)               AS gold_rows,
       count(DISTINCT nutrient)                AS distinct_nutrients,
       count(DISTINCT reaction)                AS distinct_reactions,
       count(DISTINCT odor)                    AS distinct_odors
FROM odor_gold_map;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

gold_rows,distinct_nutrients,distinct_reactions,distinct_odors
2805,438,8,43


In [8]:
%%sql
SELECT * FROM odor_gold_map LIMIT 8;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

custom_id,nutrient,reaction,odor
item-00045,"alcohol, ethyl",protein degradation,animal
item-00078,mufa 22:1 n-11,ribonucleotide degradation,fatty
item-00097,tocopherols and tocotrienols,maillard reactions,fatty
item-00099,tocopherols and tocotrienols,lipid oxidation,fatty
item-00104,tocopherols and tocotrienols,vitamin degradation,woody
item-00107,organic acids,lipid oxidation,fatty
item-00119,arginine,pigment degradation,animal
item-00121,"phytosterols, other",maillard reactions,nutty


## 3. Build the long-format odor vector (pure SQL)

In [9]:
%%sql
-- Long-format odor vector entries.
-- Faithful to the legacy odor_vector_npy_export.ipynb:
--   * nutrients per recipe form a SET (DISTINCT), duplicates across ingredients collapse;
--   * reactions per recipe form a SET of the methods' mapped reactions;
--   * every (nutrient, reaction) pair hit in the gold map contributes its odor list,
--     multiplicities included;
--   * the denominator counts EVERY odor returned (even outside the 27-odor axis);
--     odors outside the axis simply get no column.
CREATE OR REPLACE TEMP VIEW odor_long AS
WITH ing AS (
    SELECT r.rowid AS recipe_idx, lower(trim(u.ing)) AS ing
    FROM silver.RecipeZenodo r, UNNEST(r.Ingredients) AS u(ing)
),
rec_nut AS (
    SELECT DISTINCT sub.recipe_idx, lower(trim(sub.nut)) AS nut
    FROM (
        SELECT i2.recipe_idx, UNNEST(m.nutrients) AS nut
        FROM ing i2
        JOIN read_json(
            '/workspace/old_code/12_odor_vector/data/dirty_ingredient_nutrient_mapping.json',
            columns = {input: 'VARCHAR', nutrients: 'VARCHAR[]'}) m
          ON lower(trim(m.input)) = i2.ing
    ) sub
),
rec_rxn AS (
    SELECT DISTINCT r.rowid AS recipe_idx, lower(trim(cr.reaction)) AS rx
    FROM silver.RecipeZenodo r, UNNEST(r."Cooking Methods") AS u(m)
    JOIN cooking_reactions cr ON cr.method = lower(trim(u.m))
),
pairs AS (
    SELECT n.recipe_idx, g.odor
    FROM rec_nut n
    JOIN rec_rxn x ON n.recipe_idx = x.recipe_idx
    JOIN odor_gold_map g ON g.nutrient = n.nut AND g.reaction = x.rx
),
tally AS (
    SELECT recipe_idx, odor, COUNT(*) AS cnt,
           SUM(COUNT(*)) OVER (PARTITION BY recipe_idx) AS total
    FROM pairs
    GROUP BY recipe_idx, odor
)
SELECT t.recipe_idx, o.idx AS odor_idx, t.cnt::DOUBLE / t.total AS share
FROM tally t
JOIN odor_axis o ON o.odor = t.odor;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

Count


In [10]:
%%sql
SELECT count(*)                  AS long_rows,
       count(DISTINCT recipe_idx) AS recipes_with_odor_signal
FROM odor_long;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

long_rows,recipes_with_odor_signal
984344,37467


## 4. Pivot to a dense matrix and save `.npy`

In [11]:
import numpy as np
import os

OUT = '/workspace/7_create_npy_files/recipe_odor_vectors.npy'

df = raw.execute('SELECT recipe_idx, odor_idx, share FROM odor_long').df()
n_recipes = raw.execute('SELECT count(*) FROM silver.RecipeZenodo').fetchone()[0]
n_odors   = raw.execute('SELECT count(*) FROM odor_axis').fetchone()[0]

odo = np.zeros((n_recipes, n_odors), dtype=np.float32)
odo[df['recipe_idx'].to_numpy(np.int64), df['odor_idx'].to_numpy(np.int64)] = df['share'].to_numpy(np.float32)

np.save(OUT, odo)
print('saved:', OUT, '| shape:', odo.shape, '| dtype:', odo.dtype,
      '| size:', round(os.path.getsize(OUT) / 1e6, 2), 'MB')

saved: /workspace/7_create_npy_files/recipe_odor_vectors.npy | shape: (50511, 27) | dtype: float32 | size: 5.46 MB


## 5. Verification

**(a) gold cross-check** — the SQL-derived gold vs the legacy `gold_reactions.csv`.

**(b) internal** — row sums ≤ 1 and a spot-check.

**(c) vs the legacy artifact** — full row-level comparison under the documented 2-row drop
alignment.

In [12]:
import numpy as np
import pandas as pd

# ---- (a) gold cross-check vs legacy gold_reactions.csv ----
old_gold = pd.read_csv(
    '/workspace/old_code/09_gold_data_create/data/gold_reactions.csv', index_col=0)
new_gold = raw.execute("""
    SELECT custom_id,
           list(DISTINCT odor ORDER BY odor) AS gold_list
    FROM odor_gold_map GROUP BY custom_id
""").df()
print('legacy gold rows:', len(old_gold),
      '| non-empty:', int((old_gold.final_output != '[]').sum()))
print('SQL gold rows (non-empty):', len(new_gold))

def norm(s):
    return tuple(sorted(x.strip().lower() for x in eval(s))) if s != '[]' else tuple()
old_gold['g'] = old_gold.final_output.apply(norm)
merged = old_gold.merge(new_gold, on='custom_id', how='left')
new_set = merged.gold_list.apply(lambda v: tuple(v) if isinstance(v, (list, np.ndarray)) else tuple())
agree = (merged.g == new_set).sum()
print(f'rows with identical odor sets: {agree} / {len(merged)}')

legacy gold rows: 3792 | non-empty: 2805
SQL gold rows (non-empty): 2805
rows with identical odor sets: 3792 / 3792


In [13]:
import numpy as np

# ---- (b) internal ----
row_sums = odo.sum(axis=1)
print('rows with signal (sum > 0):', int((row_sums > 0).sum()))
print('max row sum (must be <= 1 + 1e-5):', round(float(row_sums.max()), 6))

name, ings, methods = raw.execute(
    "SELECT Name, Ingredients, \"Cooking Methods\" FROM silver.RecipeZenodo WHERE rowid = 0").fetchone()
r0 = odo[0]
print(f'spot-check rowid 0 {name!r}: methods = {methods}')
print('  nonzero odor shares:', {int(i): round(float(r0[i]), 4) for i in np.nonzero(r0)[0]})

rows with signal (sum > 0): 37467
max row sum (must be <= 1 + 1e-5): 1.0
spot-check rowid 0 'Pineapple Glaze for Ham': methods = ['microwave']
  nonzero odor shares: {0: 0.0099, 2: 0.1281, 3: 0.0099, 5: 0.0099, 6: 0.0197, 8: 0.0739, 9: 0.0296, 10: 0.0099, 11: 0.0049, 12: 0.0296, 13: 0.2266, 14: 0.0443, 15: 0.0049, 17: 0.0049, 18: 0.0049, 19: 0.1182, 20: 0.0049, 21: 0.0739, 22: 0.0148, 24: 0.0099, 25: 0.0394, 26: 0.1133}


In [14]:
import numpy as np

# ---- (c) vs legacy artifact (read-only, mmap) ----
OLD = '/workspace/old_code/12_odor_vector/data/recipe_odor_vectors.npy'
old = np.load(OLD, mmap_mode='r')
print('legacy shape:', old.shape, '| new shape:', odo.shape)

old_idx = np.empty(odo.shape[0], dtype=np.int64)
_o = 0
for _j in range(odo.shape[0]):
    while _o in (21475, 23805):   # old rows dropped by the new bronze ingestion
        _o += 1
    old_idx[_j] = _o
    _o += 1
old_rows = np.asarray(old[old_idx], dtype=np.float32)

row_max = np.abs(old_rows - odo).max(axis=1)
same = row_max <= 1e-6
print('identical rows:', int(same.sum()), '/', odo.shape[0],
      f'({same.mean() * 100:.3f} %)')
mism = np.nonzero(~same)[0]
print('differing new-row indices:', mism.tolist())
print('known cause: ingredient-string level differences between the legacy silver CSV')
print('and the new bronze ingestion (e.g. "peach  halved" with a double space), which')
print('change the per-recipe nutrient sets.')

legacy shape: (50513, 27) | new shape: (50511, 27)


identical rows: 50509 / 50511 (99.996 %)
differing new-row indices: [5688, 42430]
known cause: ingredient-string level differences between the legacy silver CSV
and the new bronze ingestion (e.g. "peach  halved" with a double space), which
change the per-recipe nutrient sets.


## 6. Odor entropy — `recipe_odor_entropy.npy`

The legacy `old_code/12_odor_vector/odor_entropy_get.ipynb` derives one **Shannon entropy
(base 2)** per recipe from the odor matrix: `p = row[row > 0]`, rows without any signal →
`0.0`, result cast to `float32` (1-D, one value per recipe). The exact same function is
reused here verbatim, so values stay bit-identical to the legacy pipeline. The maximum
possible entropy is `log2(27) ≈ 4.7549` (uniform row).

In [15]:
import numpy as np
import os

ENT_OUT = '/workspace/7_create_npy_files/recipe_odor_entropy.npy'

# Verbatim legacy function (old_code/12_odor_vector/odor_entropy_get.ipynb)
def calculate_entropy_1d(vector):
    p = vector[vector > 0]           # avoid log2(0)
    if len(p) == 0:
        return 0.0
    return -np.sum(p * np.log2(p))   # Shannon entropy, base 2

recipe_entropies = np.apply_along_axis(calculate_entropy_1d, axis=1, arr=odo).astype(np.float32)

np.save(ENT_OUT, recipe_entropies)
print('saved:', ENT_OUT, '| shape:', recipe_entropies.shape, '| dtype:', recipe_entropies.dtype,
      '| size:', round(os.path.getsize(ENT_OUT) / 1e6, 2), 'MB')
print('first 5 values:', recipe_entropies[:5])

saved: /workspace/7_create_npy_files/recipe_odor_entropy.npy | shape: (50511,) | dtype: float32 | size: 0.2 MB
first 5 values: [3.5427604 3.5275729 3.5448823 3.5420349 3.5876312]


In [16]:
import numpy as np

# ---- (a) internal ----
print('entropy range: [0.0,', round(float(recipe_entropies.max()), 4),
      ']  (max possible log2(27) =', round(float(np.log2(27)), 4), ')')
zero_entropy_rows = int((recipe_entropies == 0).sum())
no_signal_rows = int((odo.sum(axis=1) == 0).sum())
print('rows with entropy == 0:', zero_entropy_rows,
      '| rows with no odor signal:', no_signal_rows,
      '| consistent:', zero_entropy_rows == no_signal_rows)

# ---- (b) vs legacy artifact (read-only, mmap) ----
OLD = '/workspace/old_code/12_odor_vector/data/recipe_odor_entropy.npy'
old_entropy = np.load(OLD, mmap_mode='r')
print('legacy shape:', old_entropy.shape, '| new shape:', recipe_entropies.shape)

old_idx = np.empty(recipe_entropies.shape[0], dtype=np.int64)
_o = 0
for _j in range(recipe_entropies.shape[0]):
    while _o in (21475, 23805):   # old rows dropped by the new bronze ingestion
        _o += 1
    old_idx[_j] = _o
    _o += 1
old_vals = np.asarray(old_entropy[old_idx], dtype=np.float32)

same = np.abs(old_vals - recipe_entropies) <= 1e-6
print('identical values:', int(same.sum()), '/', recipe_entropies.shape[0],
      f'({same.mean() * 100:.3f} %)')
print('differing new-row indices:', np.nonzero(~same)[0].tolist(),
      '-> the same recipes whose odor rows differ (section 5)')

entropy range: [0.0, 3.8634 ]  (max possible log2(27) = 4.7549 )
rows with entropy == 0: 13044 | rows with no odor signal: 13044 | consistent: True
legacy shape: (50513,) | new shape: (50511,)
identical values: 50509 / 50511 (99.996 %)
differing new-row indices: [5688, 42430] -> the same recipes whose odor rows differ (section 5)


## 7. Disconnect

Close the jupysql connection; the `TEMP` views die with the session, leaving the
database file untouched.

In [17]:
%sql --close duckdb:////workspace/ICOCO.duckdb